# Testes do agent.py
Execute célula por célula para validar cada parte do agente.

In [ ]:
import anthropic, os
from dotenv import load_dotenv
load_dotenv()

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
models = client.models.list()

# Ordena a lista de modelos pelo atributo 'id' e imprime ordenado
model_ids = sorted([m.id for m in models.data])
for mid in model_ids:
    print(mid)

claude-haiku-4-5-20251001
claude-opus-4-1-20250805
claude-opus-4-5-20251101
claude-opus-4-6
claude-opus-4-7
claude-sonnet-4-5-20250929
claude-sonnet-4-6


In [ ]:

import pkg_resources

def listar_modulos_instalados():
    """
    Lista os módulos instalados no kernel Python atual com suas versões.
    """
    installed_packages = sorted(["%s==%s" % (d.project_name, d.version) for d in pkg_resources.working_set])
    for pkg in installed_packages:
        print(pkg)

# Exemplo de uso:
listar_modulos_instalados()

In [3]:
import sys
from pathlib import Path

# Garante que o Python encontra o módulo agent.py
sys.path.insert(0, str(Path(".").resolve()))

from agent import _parse_yaml_header, _build_skills_catalog, _select_skills, load_skills, run_agent

## 1. Testa `_parse_yaml_header`
Verifica se o cabeçalho YAML é extraído corretamente.

In [4]:
# Caso feliz: cabeçalho válido
md_valido = """---
name: minha-skill
description: Faz alguma coisa útil.
---

# Conteúdo da skill
"""

resultado = _parse_yaml_header(md_valido)
print("Cabeçalho extraído:", resultado)
assert resultado["name"] == "minha-skill"
assert resultado["description"] == "Faz alguma coisa útil."
print("✓ cabeçalho válido OK")

Cabeçalho extraído: {'name': 'minha-skill', 'description': 'Faz alguma coisa útil.'}
✓ cabeçalho válido OK


In [5]:
# Caso sem cabeçalho: deve retornar None
md_sem_header = "# Skill sem cabeçalho\nConteúdo direto."

resultado = _parse_yaml_header(md_sem_header)
print("Resultado esperado None:", resultado)
assert resultado is None
print("✓ sem cabeçalho OK")

Resultado esperado None: None
✓ sem cabeçalho OK


## 2. Testa `_build_skills_catalog`
Verifica se as skills reais do projeto são descobertas corretamente.

In [6]:
catalog = _build_skills_catalog()

print(f"Skills encontradas: {len(catalog)}\n")
for name, (description, path) in catalog.items():
    print(f"  [{name}]")
    print(f"    descrição: {description[:80]}...")
    print(f"    path:      {path}")
    print()

Skills encontradas: 4

  [analyzing-time-series]
    descrição: Comprehensive diagnostic analysis of time series data. Use when users provide CS...
    path:      C:/Users/Leal/AI_Projects/analytics_content_agent/skills/analyzing-time-series/SKILL.md

  [crm-omnichannel-analysis]
    descrição: Unified diagnostic analysis of CRM marketing data. Integrates online (Google Ana...
    path:      C:/Users/Leal/AI_Projects/analytics_content_agent/skills/crm-omnichannel-analysis/SKILL.md

  [docx]
    descrição: Use this skill whenever the user wants to create, read, edit, or manipulate Word...
    path:      C:/Users/Leal/AI_Projects/analytics_content_agent/skills/docx/SKILL.md

  [generic-test-skill]
    descrição: Generic test skill for validating local skill discovery and planning behavior. U...
    path:      C:/Users/Leal/AI_Projects/analytics_content_agent/skills/generic-test-skill/SKILL.md



## 3. Testa `_select_skills`
Verifica se o LLM leve seleciona as skills corretas para um prompt.

In [1]:
from agent import _build_skills_catalog, _select_skills
# Prompt que deve selecionar a skill de série temporal
catalog = _build_skills_catalog()
prompt_ts = "preciso analisar uma série temporal de vendas mensais"
selecionadas = _select_skills(prompt_ts, catalog)
print(f"Prompt: '{prompt_ts}'")
print(f"Skills selecionadas: {selecionadas}")

Prompt: 'preciso analisar uma série temporal de vendas mensais'
Skills selecionadas: ['analyzing-time-series']


In [2]:
# Prompt fora do escopo: deve retornar lista vazia
prompt_fora = "crie um logo para minha empresa"
selecionadas = _select_skills(prompt_fora, catalog)
print(f"Prompt: '{prompt_fora}'")
print(f"Skills selecionadas: {selecionadas}")
assert selecionadas == [], "Esperava lista vazia para prompt fora do escopo"
print("✓ nenhuma skill selecionada OK")

Prompt: 'crie um logo para minha empresa'
Skills selecionadas: []
✓ nenhuma skill selecionada OK


## 4. Testa `load_skills`
Verifica se o conteúdo completo das skills é carregado corretamente.

In [ ]:
# Usa o primeiro nome disponível no catálogo
from agent import _build_skills_catalog, load_skills
# Prompt que deve selecionar a skill de série temporal
catalog = _build_skills_catalog()

if catalog:
    nome = list(catalog.keys())[0]
    system_prompt = load_skills([nome])
    print(f"Skill carregada: {nome}")
    print(f"Tamanho do system prompt: {len(system_prompt)} caracteres")
    print(f"Prévia:\n{system_prompt[:300]}...")
else:
    print("Nenhuma skill encontrada no catálogo.")

Skill carregada: analyzing-time-series
Tamanho do system prompt: 2840 caracteres
Prévia:
<skill name='analyzing-time-series'>
---
name: analyzing-time-series
description: Comprehensive diagnostic analysis of time series data. Use when users provide CSV time series data and want to understand its characteristics before forecasting - stationarity, seasonality, trend, forecastability, and ...


## 5. Testa `run_agent` completo
Roda o agente de ponta a ponta com um prompt simples.

> **Atenção**: esta célula faz chamadas reais à API e ao Docker.

In [4]:
import sys
from pathlib import Path
ROOT = Path(r"c:\Users\Leal\AI_Projects\analytics_content_agent")
sys.path.insert(0, str(ROOT))
from agent import run_agent

prompt = """
    I want that you create a report showing me an time seris analysis using th csv that I import to you
"""
run_agent(
    prompt=prompt,
    model_name="claude-sonnet-4-5-20250929",
    verbose=True
)

selecting the ideal skill to execute resolve your task

[Skills selecionadas automaticamente: nenhuma]

loading the content of the selected skills
sending the message to the Claude
[stop_reason: end_turn]
I'd be happy to help you create a time series analysis report! However, I don't see any CSV file uploaded yet. 

Could you please provide the CSV file you'd like me to analyze? You can share it by:
1. Uploading the CSV file directly
2. Pasting the content of the CSV file
3. Providing the path to the CSV file if it's already in the workspace

Once you provide the data, I'll create a comprehensive time series analysis report that may include:
- Data exploration and summary statistics
- Time series visualization
- Trend analysis
- Seasonality detection
- Statistical tests
- Forecasting (if applicable)
- Key insights and findings

Please share your CSV file and let me know if there are any specific aspects of time series analysis you're particularly interested in!


In [6]:
import os
print("set?" , "ANTHROPIC_API_KEY" in os.environ)
k = os.environ.get("ANTHROPIC_API_KEY", "")
print("len:", len(k), "prefix:", k[:7] if len(k) >= 7 else repr(k))

set? True
len: 108 prefix: sk-ant-


## Test an expecific skki